In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"
OUTPUT_CSV = Path("data") / "exposure.csv"

df = pd.read_csv(INPUT_CSV)

# ---------------------------------------------------------
# CLEAN KEYS (VERY IMPORTANT)
# ---------------------------------------------------------

df["district"] = df["district"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

# ---------------------------------------------------------
# Z-SCORE
# ---------------------------------------------------------

def zscore(x):
    std = x.std(ddof=0)
    return pd.Series(0, index=x.index) if std == 0 else (x - x.mean()) / std

# ---------------------------------------------------------
# CLASSIFICATION (1–5)
# ---------------------------------------------------------

def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5

# ---------------------------------------------------------
# AGGREGATION (BLOCK → DISTRICT)
# ---------------------------------------------------------

district_df = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(district_population=("sum_population", "sum"))
)

# ---------------------------------------------------------
# MONTHWISE Z-SCORE
# ---------------------------------------------------------

district_df["pop_z"] = (
    district_df.groupby("timeperiod")["district_population"]
    .transform(zscore)
)

# ---------------------------------------------------------
# EXPOSURE CLASS
# ---------------------------------------------------------

district_df["exposure"] = district_df["pop_z"].apply(classify)

# ---------------------------------------------------------
# SAVE DISTRICT OUTPUT
# ---------------------------------------------------------

district_df.to_csv(OUTPUT_CSV, index=False)

# ---------------------------------------------------------
# MERGE BACK SAFELY (REMOVE OLD COLUMN FIRST)
# ---------------------------------------------------------

master = df.copy()

if "exposure" in master.columns:
    master = master.drop(columns=["exposure"])

master = master.merge(
    district_df[["district", "timeperiod", "exposure"]],
    on=["district", "timeperiod"],
    how="left"
)

# ---------------------------------------------------------
# SAVE MASTER (OVERWRITE)
# ---------------------------------------------------------

master.to_csv(INPUT_CSV, index=False)

print("Updated master saved:", INPUT_CSV)
print("Missing exposure:", master["exposure"].isna().sum())

print("\nPreview:")
print(master[["object_id", "district", "timeperiod", "exposure"]].head())

Updated master saved: data/MASTER_VARIABLES.csv
Missing exposure: 0

Preview:
      object_id district timeperiod  exposure
0  21-384-03276   Anugul    2023_01         3
1  21-384-03277   Anugul    2023_01         3
2  21-384-03278   Anugul    2023_01         3
3  21-384-03279   Anugul    2023_01         3
4  21-384-03280   Anugul    2023_01         3
